# 15. Spell Correction in NLP

## Overview

Spell correction identifies a potentially misspelled word and suggests a more likely alternative. A common approach is to generate candidate words and rank them using edit distance, word frequency, or a language model.

This notebook builds a small, transparent spell-correction workflow using Python and NLTK concepts. It demonstrates edit distance, candidate generation, frequency-based ranking, and context-aware limitations.

## Learning Objectives

By the end of this notebook, you will understand:
- How edit distance measures differences between words.
- How to generate correction candidates from a vocabulary.
- How to rank candidates using word frequency.
- Why the closest spelling is not always the correct word.
- How context can help distinguish plausible corrections.

## 1. Prepare a Small Vocabulary

A spell corrector needs a vocabulary of known words. In a production system, this vocabulary would usually be built from a large corpus or a domain-specific lexicon. Here, a compact vocabulary keeps the example easy to inspect.

In [1]:
from collections import Counter

training_text = """
the cat sat on the mat
the cat ate fish
the dog sat on the rug
the dog ate food
natural language processing uses language data
language models learn patterns from text
spell correction helps fix misspelled words
"""

vocabulary_tokens = training_text.lower().split()
word_counts = Counter(vocabulary_tokens)
vocabulary = set(word_counts)

print("Vocabulary size:", len(vocabulary))
print("Most common words:", word_counts.most_common(10))

Vocabulary size: 26
Most common words: [('the', 6), ('language', 3), ('cat', 2), ('sat', 2), ('on', 2), ('ate', 2), ('dog', 2), ('mat', 1), ('fish', 1), ('rug', 1)]


The example vocabulary is intentionally small. Real spell-correction systems should use a larger and more representative corpus, and should handle punctuation and tokenization carefully.

## 2. Edit Distance

**Levenshtein distance** is the minimum number of single-character insertions, deletions, and substitutions required to transform one string into another. A smaller distance usually indicates a closer spelling.

For example, changing `cat` to `cut` requires one substitution.

In [2]:
def levenshtein_distance(word_a, word_b):
    """Calculate the Levenshtein edit distance between two strings."""
    if len(word_a) < len(word_b):
        word_a, word_b = word_b, word_a

    previous_row = list(range(len(word_b) + 1))

    for i, char_a in enumerate(word_a, start=1):
        current_row = [i]
        for j, char_b in enumerate(word_b, start=1):
            insert_cost = current_row[j - 1] + 1
            delete_cost = previous_row[j] + 1
            substitute_cost = previous_row[j - 1] + (char_a != char_b)
            current_row.append(min(insert_cost, delete_cost, substitute_cost))
        previous_row = current_row

    return previous_row[-1]

print("Distance(cat, cut):", levenshtein_distance("cat", "cut"))
print("Distance(language, languge):", levenshtein_distance("language", "languge"))
print("Distance(spell, speell):", levenshtein_distance("spell", "speell"))

Distance(cat, cut): 1
Distance(language, languge): 1
Distance(spell, speell): 1


The distance is a useful signal, but it does not determine the intended word by itself. Several candidates may have the same edit distance, so a correction system needs a ranking strategy.

## 3. Generate Candidate Corrections

A simple candidate generator selects vocabulary words whose edit distance from the misspelled input is no greater than a chosen threshold. Larger thresholds find more candidates but can also introduce unrelated words.

In [3]:
def generate_candidates(word, vocabulary, max_distance=2):
    """Return vocabulary words within max_distance edits of word."""
    word = word.lower()
    candidates = []

    for candidate in vocabulary:
        distance = levenshtein_distance(word, candidate)
        if distance <= max_distance:
            candidates.append((candidate, distance))

    return sorted(candidates, key=lambda item: (item[1], item[0]))

print(generate_candidates("languge", vocabulary, max_distance=2))
print(generate_candidates("langauge", vocabulary, max_distance=2))

[('language', 1)]
[('language', 2)]


## 4. Rank Candidates by Word Frequency

If two candidates are similarly close to the input, the more frequent word may be a better suggestion. This baseline ranks first by edit distance and then by corpus frequency.

In [4]:
def rank_candidates(word, vocabulary, word_counts, max_distance=2, top_k=5):
    """Rank spelling candidates by edit distance and corpus frequency."""
    candidates = generate_candidates(word, vocabulary, max_distance=max_distance)
    ranked = sorted(
        candidates,
        key=lambda item: (item[1], -word_counts[item[0]], item[0])
    )
    return [
        {"candidate": candidate, "edit_distance": distance, "frequency": word_counts[candidate]}
        for candidate, distance in ranked[:top_k]
    ]

rank_candidates("languge", vocabulary, word_counts)

[{'candidate': 'language', 'edit_distance': 1, 'frequency': 3}]

The ranking strategy is deliberately simple: spelling distance has priority, and frequency breaks ties. A system can instead combine distance and frequency in a weighted score, especially when the most frequent word is not necessarily the intended correction.

## 5. A Combined Score

One alternative is to calculate a score that rewards frequent words while penalizing large edit distances. This is a heuristic, not a calibrated probability.

In [5]:
import math

def score_candidates(word, vocabulary, word_counts, max_distance=2, distance_penalty=1.5):
    """Score candidates using log frequency and an edit-distance penalty."""
    scored = []
    for candidate, distance in generate_candidates(word, vocabulary, max_distance):
        score = math.log(word_counts[candidate] + 1) - distance_penalty * distance
        scored.append((candidate, distance, word_counts[candidate], score))

    return sorted(scored, key=lambda row: (-row[3], row[1], row[0]))

for candidate, distance, frequency, score in score_candidates("languge", vocabulary, word_counts):
    print(f"{candidate:<15} distance={distance} frequency={frequency} score={score:.3f}")

language        distance=1 frequency=3 score=-0.114


## 6. Correct a Word

The following function returns the top-ranked suggestion. If no vocabulary word is within the maximum edit distance, it keeps the original input rather than making an unsupported guess.

In [6]:
def correct_word(word, vocabulary, word_counts, max_distance=2):
    """Return the best candidate, or the original word if none is found."""
    candidates = rank_candidates(
        word,
        vocabulary,
        word_counts,
        max_distance=max_distance,
        top_k=1,
    )

    if not candidates:
        return word

    return candidates[0]["candidate"]

misspelled_words = ["languge", "langauge", "proccessing", "speling"]

for word in misspelled_words:
    print(f"{word:<15} -> {correct_word(word, vocabulary, word_counts)}")

languge         -> language
langauge        -> language
proccessing     -> processing
speling         -> speling


## 7. Context Matters

Word-level spelling correction can fail when multiple valid words have similar spellings. Context helps determine the intended word. For example, `their` and `there` are both valid words, but they fit different sentence contexts.

A frequency-and-edit-distance baseline cannot reliably resolve these cases because it scores words in isolation. Context-aware correction can use surrounding tokens, n-gram language models, or neural language models.

In [7]:
context_examples = [
    "Put the book over there.",
    "Their project uses natural language processing.",
]

for sentence in context_examples:
    print(sentence)

Put the book over there.
Their project uses natural language processing.


The sentences above illustrate why spelling correction is not only a character-level problem. Both `their` and `there` are correctly spelled, so the system needs grammatical and semantic context rather than edit distance alone.

## 8. Practical Considerations

- **Vocabulary coverage:** a small vocabulary may incorrectly flag valid but rare words.
- **Tokenization:** punctuation and contractions need consistent handling.
- **Capitalization:** names and sentence-initial words may require special treatment.
- **Language and domain:** vocabulary and error patterns differ across languages and topics.
- **Performance:** comparing a query with every vocabulary word can be expensive for large dictionaries; candidate-generation indexes can make it faster.
- **Evaluation:** a correction system should be tested on realistic misspellings with metrics such as correction accuracy and top-k accuracy.

## 9. Summary

- Edit distance measures the character-level changes between strings.
- Candidate generation finds vocabulary words close to a suspected misspelling.
- Word frequency can help rank candidates, but it is not enough for every case.
- Context is necessary to distinguish valid words that are used incorrectly.
- Real-world systems need larger vocabularies, robust preprocessing, and evaluation on representative data.